# Options Pricing: Black-Scholes & Greeks

**Category:** 11-Financial  
**Description:** Option valuation, Greeks, and risk analysis  
**Libraries:** NumPy, SciPy, Matplotlib

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q numpy scipy matplotlib pandas

# Model aliases
CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
GPT = "openai-chat:gpt-5"

MODEL = CLAUDE

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')

---

# Part 1: Black-Scholes Model

---

In [ ]:
class BlackScholes:
    """
    Black-Scholes option pricing model.
    
    Parameters:
        S: Current stock price
        K: Strike price
        T: Time to expiration (years)
        r: Risk-free interest rate
        sigma: Volatility (annualized)
    """
    
    def __init__(self, S, K, T, r, sigma):
        self.S = S
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        
        # Calculate d1 and d2
        self.d1 = (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        self.d2 = self.d1 - sigma * np.sqrt(T)
    
    def call_price(self):
        """Calculate call option price."""
        return (self.S * norm.cdf(self.d1) - 
                self.K * np.exp(-self.r * self.T) * norm.cdf(self.d2))
    
    def put_price(self):
        """Calculate put option price."""
        return (self.K * np.exp(-self.r * self.T) * norm.cdf(-self.d2) - 
                self.S * norm.cdf(-self.d1))
    
    # Greeks
    def delta_call(self):
        return norm.cdf(self.d1)
    
    def delta_put(self):
        return norm.cdf(self.d1) - 1
    
    def gamma(self):
        return norm.pdf(self.d1) / (self.S * self.sigma * np.sqrt(self.T))
    
    def theta_call(self):
        term1 = -self.S * norm.pdf(self.d1) * self.sigma / (2 * np.sqrt(self.T))
        term2 = -self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(self.d2)
        return (term1 + term2) / 365  # Daily theta
    
    def theta_put(self):
        term1 = -self.S * norm.pdf(self.d1) * self.sigma / (2 * np.sqrt(self.T))
        term2 = self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(-self.d2)
        return (term1 + term2) / 365
    
    def vega(self):
        return self.S * norm.pdf(self.d1) * np.sqrt(self.T) / 100  # Per 1% vol change
    
    def rho_call(self):
        return self.K * self.T * np.exp(-self.r * self.T) * norm.cdf(self.d2) / 100
    
    def rho_put(self):
        return -self.K * self.T * np.exp(-self.r * self.T) * norm.cdf(-self.d2) / 100

In [ ]:
# Example: AAPL-like option
S = 180      # Current price
K = 185      # Strike price
T = 30/365   # 30 days to expiration
r = 0.05     # 5% risk-free rate
sigma = 0.25 # 25% implied volatility

option = BlackScholes(S, K, T, r, sigma)

print("="*50)
print("BLACK-SCHOLES OPTION PRICING")
print("="*50)
print(f"\nUnderlying: ${S:.2f}")
print(f"Strike:     ${K:.2f}")
print(f"Days to Exp: {T*365:.0f}")
print(f"Risk-free:  {r:.1%}")
print(f"Volatility: {sigma:.1%}")
print("\n" + "-"*50)
print(f"Call Price:  ${option.call_price():.2f}")
print(f"Put Price:   ${option.put_price():.2f}")
print("\n" + "-"*50)
print("GREEKS (Call)")
print(f"  Delta:  {option.delta_call():.4f}")
print(f"  Gamma:  {option.gamma():.4f}")
print(f"  Theta:  ${option.theta_call():.4f}/day")
print(f"  Vega:   ${option.vega():.4f}/1% vol")
print(f"  Rho:    ${option.rho_call():.4f}/1% rate")

In [ ]:
%%ai $MODEL
Explain the Greeks in options trading:

1. Delta: What does a delta of 0.4 mean practically?
2. Gamma: Why is it important for hedging?
3. Theta: How does time decay work?
4. Vega: How does volatility affect option prices?

Give practical examples a trader would understand.

---

# Part 2: Option Price Sensitivity

---

In [ ]:
# Price sensitivity to underlying
stock_prices = np.linspace(150, 210, 100)
call_prices = [BlackScholes(s, K, T, r, sigma).call_price() for s in stock_prices]
put_prices = [BlackScholes(s, K, T, r, sigma).put_price() for s in stock_prices]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Option prices
axes[0].plot(stock_prices, call_prices, 'b-', linewidth=2, label='Call')
axes[0].plot(stock_prices, put_prices, 'r-', linewidth=2, label='Put')
axes[0].axvline(K, color='gray', linestyle='--', label=f'Strike ${K}')
axes[0].axvline(S, color='green', linestyle=':', label=f'Current ${S}')
axes[0].set_title('Option Price vs Stock Price')
axes[0].set_xlabel('Stock Price ($)')
axes[0].set_ylabel('Option Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Delta
call_deltas = [BlackScholes(s, K, T, r, sigma).delta_call() for s in stock_prices]
put_deltas = [BlackScholes(s, K, T, r, sigma).delta_put() for s in stock_prices]

axes[1].plot(stock_prices, call_deltas, 'b-', linewidth=2, label='Call Delta')
axes[1].plot(stock_prices, put_deltas, 'r-', linewidth=2, label='Put Delta')
axes[1].axvline(K, color='gray', linestyle='--', label=f'Strike ${K}')
axes[1].axhline(0.5, color='gray', linestyle=':', alpha=0.5)
axes[1].set_title('Delta vs Stock Price')
axes[1].set_xlabel('Stock Price ($)')
axes[1].set_ylabel('Delta')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Time decay visualization
times_to_exp = [60/365, 30/365, 14/365, 7/365, 1/365]  # Days
colors = plt.cm.viridis(np.linspace(0, 0.8, len(times_to_exp)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for t, color in zip(times_to_exp, colors):
    prices = [BlackScholes(s, K, t, r, sigma).call_price() for s in stock_prices]
    thetas = [BlackScholes(s, K, t, r, sigma).theta_call() for s in stock_prices]
    
    axes[0].plot(stock_prices, prices, color=color, linewidth=2, 
                 label=f'{int(t*365)} days')
    axes[1].plot(stock_prices, thetas, color=color, linewidth=2,
                 label=f'{int(t*365)} days')

axes[0].axvline(K, color='gray', linestyle='--')
axes[0].set_title('Call Price vs Time to Expiration')
axes[0].set_xlabel('Stock Price ($)')
axes[0].set_ylabel('Call Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].axvline(K, color='gray', linestyle='--')
axes[1].set_title('Theta (Time Decay) by Days to Expiration')
axes[1].set_xlabel('Stock Price ($)')
axes[1].set_ylabel('Theta ($/day)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# Part 3: Implied Volatility

---

In [ ]:
def implied_volatility(market_price, S, K, T, r, option_type='call', tol=1e-6, max_iter=100):
    """
    Calculate implied volatility using Newton-Raphson method.
    """
    sigma = 0.25  # Initial guess
    
    for i in range(max_iter):
        bs = BlackScholes(S, K, T, r, sigma)
        
        if option_type == 'call':
            price = bs.call_price()
        else:
            price = bs.put_price()
        
        vega = bs.vega() * 100  # Convert back from per 1% to per 1
        
        diff = price - market_price
        
        if abs(diff) < tol:
            return sigma
        
        if vega < 1e-10:
            break
            
        sigma = sigma - diff / vega
        sigma = max(0.01, min(5.0, sigma))  # Keep sigma reasonable
    
    return sigma

# Example: Given market price, find IV
market_call_price = 3.50
iv = implied_volatility(market_call_price, S, K, T, r)
print(f"Market Call Price: ${market_call_price:.2f}")
print(f"Implied Volatility: {iv:.1%}")

# Verify
check_bs = BlackScholes(S, K, T, r, iv)
print(f"Verification Price: ${check_bs.call_price():.2f}")

In [ ]:
# Volatility Smile (IV across strikes)
strikes = np.linspace(160, 200, 15)

# Simulate market prices with skew (realistic)
base_iv = 0.25
skew = 0.002  # IV increases for lower strikes (put skew)
smile = 0.0005  # IV increases for far OTM options

market_ivs = []
for k in strikes:
    moneyness = np.log(S/k)
    iv = base_iv - skew * moneyness / 0.1 + smile * (moneyness/0.1)**2
    market_ivs.append(iv)

# Plot volatility smile
plt.figure(figsize=(10, 6))
plt.plot(strikes, [iv*100 for iv in market_ivs], 'bo-', linewidth=2, markersize=8)
plt.axvline(S, color='green', linestyle='--', label=f'Spot ${S}')
plt.axhline(base_iv*100, color='gray', linestyle=':', alpha=0.5, label=f'ATM IV {base_iv:.0%}')
plt.title('Implied Volatility Smile')
plt.xlabel('Strike Price ($)')
plt.ylabel('Implied Volatility (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

# Part 4: Option Strategies

---

In [ ]:
def plot_strategy_payoff(positions, title, stock_range=(150, 210)):
    """
    Plot option strategy payoff at expiration.
    
    positions: list of dicts with keys 'type', 'strike', 'premium', 'quantity'
    """
    S_range = np.linspace(stock_range[0], stock_range[1], 200)
    payoff = np.zeros_like(S_range)
    total_cost = 0
    
    for pos in positions:
        K = pos['strike']
        prem = pos['premium']
        qty = pos['quantity']
        total_cost += prem * qty
        
        if pos['type'] == 'call':
            payoff += qty * (np.maximum(S_range - K, 0) - prem)
        elif pos['type'] == 'put':
            payoff += qty * (np.maximum(K - S_range, 0) - prem)
        elif pos['type'] == 'stock':
            payoff += qty * (S_range - K)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(S_range, payoff, 'b-', linewidth=2)
    plt.fill_between(S_range, payoff, 0, where=(payoff >= 0), alpha=0.3, color='green')
    plt.fill_between(S_range, payoff, 0, where=(payoff < 0), alpha=0.3, color='red')
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(S, color='gray', linestyle='--', label=f'Current: ${S}')
    
    for pos in positions:
        plt.axvline(pos['strike'], color='gray', linestyle=':', alpha=0.5)
    
    plt.title(f'{title}\nMax Profit: ${payoff.max():.2f} | Max Loss: ${payoff.min():.2f}')
    plt.xlabel('Stock Price at Expiration ($)')
    plt.ylabel('Profit/Loss ($)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return payoff

In [ ]:
# Bull Call Spread
bull_call = [
    {'type': 'call', 'strike': 180, 'premium': 5.00, 'quantity': 1},   # Buy
    {'type': 'call', 'strike': 190, 'premium': 2.00, 'quantity': -1}   # Sell
]
plot_strategy_payoff(bull_call, 'Bull Call Spread (180/190)')

In [ ]:
# Iron Condor
iron_condor = [
    {'type': 'put', 'strike': 165, 'premium': 1.00, 'quantity': 1},    # Buy OTM put
    {'type': 'put', 'strike': 170, 'premium': 2.00, 'quantity': -1},   # Sell OTM put
    {'type': 'call', 'strike': 190, 'premium': 2.00, 'quantity': -1},  # Sell OTM call
    {'type': 'call', 'strike': 195, 'premium': 1.00, 'quantity': 1}    # Buy OTM call
]
plot_strategy_payoff(iron_condor, 'Iron Condor (165/170/190/195)')

In [ ]:
# Straddle
straddle = [
    {'type': 'call', 'strike': 180, 'premium': 5.00, 'quantity': 1},
    {'type': 'put', 'strike': 180, 'premium': 4.50, 'quantity': 1}
]
plot_strategy_payoff(straddle, 'Long Straddle (180 Strike)')

In [ ]:
%%ai $MODEL
Compare these three option strategies:

1. Bull Call Spread: When would you use it?
2. Iron Condor: What market conditions favor this?
3. Long Straddle: What's the trader betting on?

Include the risk/reward tradeoffs for each.

---

## Summary

This notebook covered:
- Black-Scholes option pricing model
- The Greeks (Delta, Gamma, Theta, Vega, Rho)
- Implied volatility calculation
- Volatility smile visualization
- Common option strategies (spreads, iron condors, straddles)

**Next:** Explore risk analysis with VaR in `03-risk-analysis.ipynb`

---